# U-Net flood segmentation

## 1. Environment and imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip -q install torchinfo


In [ ]:
import os
import zipfile
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as T

from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchinfo import summary


## 2. Hyperparameters

In [ ]:
SEED = 42
IMAGE_SIZE = (512, 512)
BATCH_SIZE = 8
LEARNING_RATE = 0.001
ADAM_BETAS = (0.9, 0.999)
COMMUNICATION_ROUNDS = 60
CLIENT_SIZES = [108, 51, 58]
CLIENT_STEPS = [14, 7, 8]
CLIENT_WEIGHTS = [n / sum(CLIENT_SIZES) for n in CLIENT_SIZES]



MU = 0.01

CONFIDENCE_THRESHOLD = 0.95
MAX_EMA_DECAY = 0.99
WARMUP_ROUNDS = 5
UNSUPERVISED_START_WEIGHT = 0.10
UNSUPERVISED_MAX_WEIGHT = 1.00

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Client aggregation weights:", CLIENT_WEIGHTS)


Device: cuda
Client aggregation weights: [0.4976958525345622, 0.2350230414746544, 0.2672811059907834]


## 3. Load dataset and preprocessing

In [ ]:
zip_path = '/content/drive/MyDrive/DLP_Project/flood_area.zip'
extract_path = '/content/flood_segmentation'

if not os.path.exists(os.path.join(extract_path, "metadata.csv")):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

class FloodDataset(Dataset):
    def __init__(self, base_dir, image_size=IMAGE_SIZE):
        self.base_dir = base_dir
        self.image_dir = os.path.join(base_dir, "Image")
        self.mask_dir = os.path.join(base_dir, "Mask")
        self.metadata = pd.read_csv(os.path.join(base_dir, "metadata.csv"))
        self.image_size = image_size

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.metadata.iloc[index, 0])
        mask_path = os.path.join(self.mask_dir, self.metadata.iloc[index, 1])

        image = torchvision.io.decode_image(image_path, mode="RGB")
        mask = torchvision.io.decode_image(mask_path, mode="GRAY")

        image = T.Resize(self.image_size)(image)

        mask = T.Resize(self.image_size, interpolation=T.InterpolationMode.NEAREST)(mask)
        image = image.float() / 255.0
        mask = (mask > 0).float()
        return image, mask

dataset = FloodDataset(base_dir=extract_path)
print("Total image-mask pairs:", len(dataset))


Total image-mask pairs: 290


## 4. Fixed train / validation / test split

In [ ]:

assert len(dataset) == 290, f"Expected 290 pairs, found {len(dataset)}"

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [217, 29, 44], generator=generator
)

print(f"Train: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=torch.cuda.is_available())


Train: 217
Validation: 29
Test: 44


## 5. U-Net architecture trained from scratch

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.conv(x)


class UpConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode="bilinear",
                              align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.pool = nn.MaxPool2d(2, 2)
        self.down1 = DoubleConv(in_channels, 64)
        self.down2 = DoubleConv(64, 128)
        self.down3 = DoubleConv(128, 256)
        self.down4 = DoubleConv(256, 512)
        self.down5 = DoubleConv(512, 1024)
        self.up1 = UpConv(1024, 512)
        self.up2 = UpConv(512, 256)
        self.up3 = UpConv(256, 128)
        self.up4 = UpConv(128, 64)
        self.final = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool(x1))
        x3 = self.down3(self.pool(x2))
        x4 = self.down4(self.pool(x3))
        x5 = self.down5(self.pool(x4))
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.final(x)

model_check = UNet().to(device)
print(summary(model_check, input_size=(1, 3, 512, 512), verbose=0))
del model_check
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Layer (type:depth-idx)                   Output Shape              Param #
UNet                                     [1, 1, 512, 512]          --
├─DoubleConv: 1-1                        [1, 64, 512, 512]         --
│    └─Sequential: 2-1                   [1, 64, 512, 512]         --
│    │    └─Conv2d: 3-1                  [1, 64, 512, 512]         1,728
│    │    └─BatchNorm2d: 3-2             [1, 64, 512, 512]         128
│    │    └─ReLU: 3-3                    [1, 64, 512, 512]         --
│    │    └─Conv2d: 3-4                  [1, 64, 512, 512]         36,864
│    │    └─BatchNorm2d: 3-5             [1, 64, 512, 512]         128
│    │    └─ReLU: 3-6                    [1, 64, 512, 512]         --
├─MaxPool2d: 1-2                         [1, 64, 256, 256]         --
├─DoubleConv: 1-3                        [1, 128, 256, 256]        --
│    └─Sequential: 2-2                   [1, 128, 256, 256]        --
│    │    └─Conv2d: 3-7                  [1, 128, 256, 256]        73,728
│ 

## 6. Client partitioning and labeled/unlabeled split

In [ ]:

client_generator = torch.Generator().manual_seed(SEED)
client_datasets = random_split(train_dataset, CLIENT_SIZES, generator=client_generator)



LABELED_COUNTS = [22, 10, 12]
client_labeled_datasets = []
client_unlabeled_datasets = []

for client_ds, labeled_count in zip(client_datasets, LABELED_COUNTS):
    split_gen = torch.Generator().manual_seed(SEED + len(client_labeled_datasets) + 1)
    labeled_ds, unlabeled_ds = random_split(
        client_ds, [labeled_count, len(client_ds) - labeled_count],
        generator=split_gen
    )
    client_labeled_datasets.append(labeled_ds)
    client_unlabeled_datasets.append(unlabeled_ds)

print("Client sizes:", [len(x) for x in client_datasets])
print("Labeled sizes:", [len(x) for x in client_labeled_datasets])
print("Unlabeled sizes:", [len(x) for x in client_unlabeled_datasets])


Client sizes: [108, 51, 58]
Labeled sizes: [22, 10, 12]
Unlabeled sizes: [86, 41, 46]


## 7. Shared utilities: FedProx, EMA, pooled metrics, validation

In [ ]:
criterion = nn.BCEWithLogitsLoss()
strong_color_jitter = T.ColorJitter(
    brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05
)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_loader(ds, shuffle):
    return DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=shuffle,
        num_workers=2, pin_memory=torch.cuda.is_available(),
        drop_last=False
    )


def pooled_metrics(model, loader):
    model.eval()
    tp = fp = fn = tn = 0
    total_loss = 0.0
    batches = 0
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            logits = model(images)
            total_loss += criterion(logits, masks).item()
            batches += 1
            pred = (torch.sigmoid(logits) >= 0.5)
            truth = (masks >= 0.5)
            tp += (pred & truth).sum().item()
            fp += (pred & ~truth).sum().item()
            fn += (~pred & truth).sum().item()
            tn += (~pred & ~truth).sum().item()

    eps = 1e-12
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    return {
        "loss": total_loss / max(batches, 1),
        "iou": iou, "f1": f1, "precision": precision,
        "recall": recall, "pixel_accuracy": accuracy,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }


def fedprox_penalty(model, global_parameters):

    penalty = torch.zeros((), device=device)
    for local_param, global_param in zip(model.parameters(), global_parameters):
        penalty += torch.sum((local_param - global_param) ** 2)
    return 0.5 * MU * penalty


def update_ema(teacher, student, decay):
    with torch.no_grad():
        for teacher_param, student_param in zip(teacher.parameters(), student.parameters()):
            teacher_param.data.mul_(decay).add_(student_param.data, alpha=1.0 - decay)
        for teacher_buffer, student_buffer in zip(teacher.buffers(), student.buffers()):
            if teacher_buffer.dtype.is_floating_point:
                teacher_buffer.data.mul_(decay).add_(student_buffer.data, alpha=1.0 - decay)
            else:
                teacher_buffer.data.copy_(student_buffer.data)


def unsupervised_weight(round_number):
    if round_number <= WARMUP_ROUNDS:
        return 0.0
    if round_number >= 15:
        return UNSUPERVISED_MAX_WEIGHT

    progress = (round_number - 6) / (15 - 6)
    return UNSUPERVISED_START_WEIGHT + progress * (
        UNSUPERVISED_MAX_WEIGHT - UNSUPERVISED_START_WEIGHT
    )


def aggregate_client_models(client_states):

    aggregated = copy.deepcopy(client_states[0][0])
    total_samples = sum(n for _, n in client_states)
    for key in aggregated:
        if torch.is_floating_point(aggregated[key]):
            aggregated[key] = sum(
                state[key] * (n / total_samples)
                for state, n in client_states
            )
        else:
            aggregated[key] = client_states[0][0][key]
    return aggregated


## 8. Experiment runner

In [ ]:
def train_fedprox_experiment(semi_supervised=False):
    set_seed(SEED)
    global_model = UNet().to(device)
    best_state = copy.deepcopy(global_model.state_dict())
    best_val_iou = -1.0
    best_round = None
    history = []

    if semi_supervised:
        print("\nStarting FedProx + semi-supervised student-teacher experiment")
    else:
        print("\nStarting fully supervised FedProx experiment")

    for round_number in range(1, COMMUNICATION_ROUNDS + 1):
        global_model.train()
        global_parameters = [
            p.detach().clone().to(device) for p in global_model.parameters()
        ]
        client_states = []

        for client_id in range(3):
            if semi_supervised:
                labeled_ds = client_labeled_datasets[client_id]
                unlabeled_ds = client_unlabeled_datasets[client_id]
                labeled_loader = make_loader(labeled_ds, shuffle=True)
                unlabeled_loader = make_loader(unlabeled_ds, shuffle=True)
                labeled_iter = iter(labeled_loader)
                unlabeled_iter = iter(unlabeled_loader)
                local_sample_count = len(labeled_ds) + len(unlabeled_ds)
            else:
                client_ds = client_datasets[client_id]
                local_loader = make_loader(client_ds, shuffle=True)
                local_iter = iter(local_loader)
                local_sample_count = len(client_ds)

            student = UNet().to(device)
            student.load_state_dict(global_model.state_dict())
            teacher = None
            if semi_supervised:
                teacher = UNet().to(device)
                teacher.load_state_dict(student.state_dict())
                teacher.eval()

            optimizer = optim.Adam(
                student.parameters(), lr=LEARNING_RATE, betas=ADAM_BETAS
            )
            local_steps = CLIENT_STEPS[client_id]
            current_unsup_weight = unsupervised_weight(round_number)
            student.train()

            for local_step in range(local_steps):
                optimizer.zero_grad(set_to_none=True)

                if semi_supervised:
                    try:
                        labeled_images, labeled_masks = next(labeled_iter)
                    except StopIteration:
                        labeled_iter = iter(labeled_loader)
                        labeled_images, labeled_masks = next(labeled_iter)

                    try:
                        unlabeled_images, _ = next(unlabeled_iter)
                    except StopIteration:
                        unlabeled_iter = iter(unlabeled_loader)
                        unlabeled_images, _ = next(unlabeled_iter)

                    labeled_images = labeled_images.to(device, non_blocking=True)
                    labeled_masks = labeled_masks.to(device, non_blocking=True)
                    unlabeled_images = unlabeled_images.to(device, non_blocking=True)

                    supervised_logits = student(labeled_images)
                    supervised_loss = criterion(supervised_logits, labeled_masks)

                    unsupervised_loss = torch.zeros((), device=device)
                    if current_unsup_weight > 0:
                        with torch.no_grad():
                            teacher_logits = teacher(unlabeled_images)
                            teacher_probs = torch.sigmoid(teacher_logits)
                            confidence = torch.maximum(teacher_probs, 1.0 - teacher_probs)
                            confident_pixels = confidence >= CONFIDENCE_THRESHOLD
                            pseudo_masks = (teacher_probs >= 0.5).float()

                        strong_images = torch.stack([
                            strong_color_jitter(img) for img in unlabeled_images
                        ]).clamp(0.0, 1.0)
                        student_unlabeled_logits = student(strong_images)
                        pixel_loss = F.binary_cross_entropy_with_logits(
                            student_unlabeled_logits, pseudo_masks, reduction="none"
                        )
                        confident_float = confident_pixels.float()
                        unsupervised_loss = (
                            (pixel_loss * confident_float).sum()
                            / confident_float.sum().clamp_min(1.0)
                        )

                    prox_loss = fedprox_penalty(student, global_parameters)
                    loss = supervised_loss + current_unsup_weight * unsupervised_loss + prox_loss
                else:
                    try:
                        images, masks = next(local_iter)
                    except StopIteration:
                        local_iter = iter(local_loader)
                        images, masks = next(local_iter)

                    images = images.to(device, non_blocking=True)
                    masks = masks.to(device, non_blocking=True)
                    logits = student(images)
                    supervised_loss = criterion(logits, masks)
                    prox_loss = fedprox_penalty(student, global_parameters)
                    unsupervised_loss = torch.zeros((), device=device)
                    loss = supervised_loss + prox_loss

                loss.backward()
                optimizer.step()

                if semi_supervised:
                    global_step = (round_number - 1) * sum(CLIENT_STEPS) + local_step + 1
                    ema_decay = min(MAX_EMA_DECAY, 1.0 - 1.0 / (global_step + 1.0))
                    update_ema(teacher, student, ema_decay)

            client_states.append((copy.deepcopy(student.state_dict()), local_sample_count))

            del student, optimizer
            if teacher is not None:
                del teacher
            if semi_supervised:
                del labeled_images, labeled_masks, unlabeled_images
            else:
                del images, masks

            import gc
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        global_model.load_state_dict(aggregate_client_models(client_states))

        val_metrics = pooled_metrics(global_model, val_loader)
        record = {
            "round": round_number,
            "val_loss": val_metrics["loss"],
            "val_iou": val_metrics["iou"],
            "val_f1": val_metrics["f1"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_pixel_accuracy": val_metrics["pixel_accuracy"],
            "unsupervised_weight": unsupervised_weight(round_number) if semi_supervised else 0.0
        }
        history.append(record)

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            best_round = round_number
            best_state = copy.deepcopy(global_model.state_dict())

        print(
            f"Round {round_number:02d}/{COMMUNICATION_ROUNDS} | "
            f"Val IoU: {val_metrics['iou']:.6f} | "
            f"Val F1: {val_metrics['f1']:.6f} | "
            f"Unsup weight: {record['unsupervised_weight']:.2f}"
        )

    global_model.load_state_dict(best_state)
    test_metrics = pooled_metrics(global_model, test_loader)

    print("\nBest round:", best_round)
    print("Best validation IoU:", f"{best_val_iou:.6f}")
    print("Pooled test metrics:")
    for key in ["iou", "f1", "precision", "recall", "pixel_accuracy"]:
        print(f"{key}: {test_metrics[key]:.6f}")

    return global_model, pd.DataFrame(history), {
        "best_round": best_round,
        "best_val_iou": best_val_iou,
        "test_metrics": test_metrics
    }

## 9. Experiment A — Fully supervised FedProx (no semi-supervised learning)

In [ ]:
supervised_model, supervised_history, supervised_results = train_fedprox_experiment(
    semi_supervised=False
)

supervised_history.tail()


\nStarting fully supervised FedProx experiment
Round 01/60 | Val IoU: 0.422213 | Val F1: 0.593741 | Unsup weight: 0.00
Round 02/60 | Val IoU: 0.427922 | Val F1: 0.599364 | Unsup weight: 0.00
Round 03/60 | Val IoU: 0.448616 | Val F1: 0.619372 | Unsup weight: 0.00
Round 04/60 | Val IoU: 0.539179 | Val F1: 0.700606 | Unsup weight: 0.00
Round 05/60 | Val IoU: 0.631891 | Val F1: 0.774428 | Unsup weight: 0.00
Round 06/60 | Val IoU: 0.528585 | Val F1: 0.691600 | Unsup weight: 0.00
Round 07/60 | Val IoU: 0.664154 | Val F1: 0.798188 | Unsup weight: 0.00
Round 08/60 | Val IoU: 0.684504 | Val F1: 0.812707 | Unsup weight: 0.00
Round 09/60 | Val IoU: 0.579332 | Val F1: 0.733642 | Unsup weight: 0.00
Round 10/60 | Val IoU: 0.657913 | Val F1: 0.793664 | Unsup weight: 0.00
Round 11/60 | Val IoU: 0.598223 | Val F1: 0.748610 | Unsup weight: 0.00
Round 12/60 | Val IoU: 0.684012 | Val F1: 0.812360 | Unsup weight: 0.00
Round 13/60 | Val IoU: 0.694399 | Val F1: 0.819641 | Unsup weight: 0.00
Round 14/60 | Val

,round,val_loss,val_iou,val_f1,val_precision,val_recall,val_pixel_accuracy,unsupervised_weight
55,56,0.301948,0.738027,0.849270,0.851226,0.847323,0.873074,0.0
56,57,0.336937,0.720768,0.837728,0.780656,0.903803,0.852239,0.0
57,58,0.308530,0.731328,0.844818,0.860619,0.829586,0.871386,0.0
58,59,0.324997,0.737088,0.848648,0.861698,0.835988,0.874164,0.0
59,60,0.328868,0.723255,0.839406,0.817239,0.862808,0.860679,0.0


## 10. Experiment B — Semi-supervised FedProx + Student–Teacher

In [ ]:
result1 = pd.DataFrame([
    {
        "Experiment": "Fully supervised FedProx",
        "Best round": supervised_results["best_round"],
        "Best validation IoU": supervised_results["best_val_iou"],
        "Test IoU": supervised_results["test_metrics"]["iou"],
        "Test F1": supervised_results["test_metrics"]["f1"],
        "Test precision": supervised_results["test_metrics"]["precision"],
        "Test recall": supervised_results["test_metrics"]["recall"],
        "Test pixel accuracy": supervised_results["test_metrics"]["pixel_accuracy"],
    }])

In [ ]:
result1

,Experiment,Best round,Best validation IoU,Test IoU,Test F1,Test precision,Test recall,Test pixel accuracy
0,Fully supervised FedProx,56,0.738027,0.722032,0.838581,0.848703,0.828699,0.858446


In [ ]:
import copy
import gc
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
assert torch.cuda.is_available(), (
    "Connect this Colab runtime to a GPU before running."
)

device = torch.device("cuda")

MICRO_BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 8
ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // MICRO_BATCH_SIZE

assert EFFECTIVE_BATCH_SIZE % MICRO_BATCH_SIZE == 0

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/DLP_Project/"
    "friend_unet_fedprox_student_teacher"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LAST_PATH = OUTPUT_DIR / "last_checkpoint.pt"
BEST_PATH = OUTPUT_DIR / "best_checkpoint.pt"
HISTORY_PATH = OUTPUT_DIR / "training_history.csv"

run_config = {
    "experiment": "friend_unet_fedprox_student_teacher",
    "seed": SEED,
    "image_size": list(IMAGE_SIZE),
    "physical_batch_size": MICRO_BATCH_SIZE,
    "effective_batch_size": EFFECTIVE_BATCH_SIZE,
    "accumulation_steps": ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "adam_betas": tuple(ADAM_BETAS),
    "rounds": COMMUNICATION_ROUNDS,
    "client_sizes": list(CLIENT_SIZES),
    "client_steps": list(CLIENT_STEPS),
    "mu": MU,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "max_ema_decay": MAX_EMA_DECAY,
    "warmup_rounds": WARMUP_ROUNDS,
    "precision": "cuda_amp_float16",
}

print("GPU:", torch.cuda.get_device_name(0))
print("FedProx μ:", MU)
print("Physical batch size:", MICRO_BATCH_SIZE)
print("Effective batch size:", EFFECTIVE_BATCH_SIZE)
print("Results directory:", OUTPUT_DIR)






labeled_generators = {}
unlabeled_generators = {}
labeled_loaders = {}
unlabeled_loaders = {}

for client_index in range(3):
    labeled_generators[client_index] = torch.Generator().manual_seed(
        SEED + client_index + 1
    )

    unlabeled_generators[client_index] = torch.Generator().manual_seed(
        SEED + 1001 + client_index
    )

    labeled_loaders[client_index] = DataLoader(
        client_labeled_datasets[client_index],
        batch_size=EFFECTIVE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=labeled_generators[client_index],
        drop_last=False,
    )

    unlabeled_loaders[client_index] = DataLoader(
        client_unlabeled_datasets[client_index],
        batch_size=EFFECTIVE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=unlabeled_generators[client_index],
        drop_last=False,
    )


val_loader_small = DataLoader(
    val_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

test_loader_small = DataLoader(
    test_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


def save_checkpoint_atomic(path, state):
    temporary_path = path.with_suffix(".tmp")
    torch.save(state, temporary_path)
    os.replace(temporary_path, path)


def aggregate_states_cpu(client_states, client_sizes):
    total = sum(client_sizes)
    aggregated = {}

    for name, reference in client_states[0].items():
        if reference.is_floating_point():
            value = torch.zeros_like(reference)

            for state, size in zip(client_states, client_sizes):
                value.add_(state[name], alpha=size / total)

            assert torch.isfinite(value).all(), name
            aggregated[name] = value
        else:

            aggregated[name] = reference.clone()

    return aggregated


def evaluate_unet(model, loader):
    model.eval()

    total_loss = 0.0
    sample_count = 0
    counts = torch.zeros(4, dtype=torch.int64, device=device)

    with torch.inference_mode():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = model(images)
                loss = F.binary_cross_entropy_with_logits(
                    logits, masks
                )

            batch_count = images.shape[0]
            total_loss += loss.item() * batch_count
            sample_count += batch_count

            prediction = logits >= 0
            target = masks >= 0.5

            counts += torch.stack([
                (prediction & target).sum(),
                (prediction & ~target).sum(),
                (~prediction & target).sum(),
                (~prediction & ~target).sum(),
            ])

    tp, fp, fn, tn = counts.cpu().tolist()

    return {
        "loss": total_loss / sample_count,
        "iou": tp / max(tp + fp + fn, 1),
        "precision": tp / max(tp + fp, 1),
        "recall": tp / max(tp + fn, 1),
        "f1": 2 * tp / max(2 * tp + fp + fn, 1),
        "pixel_accuracy": (tp + tn) / max(tp + fp + fn + tn, 1),
    }






def train_client(global_state, client_index, round_number):
    student = UNet().to(device)
    student.load_state_dict(global_state)
    student.train()

    teacher = UNet().to(device)
    teacher.load_state_dict(global_state)
    teacher.requires_grad_(False)
    teacher.eval()

    global_parameters = [
        parameter.detach().clone()
        for parameter in student.parameters()
    ]

    optimizer = torch.optim.Adam(
        student.parameters(),
        lr=LEARNING_RATE,
        betas=ADAM_BETAS,
    )
    scaler = torch.amp.GradScaler(
    "cuda",
    init_scale=128.0,
    )

    labeled_loader = labeled_loaders[client_index]
    unlabeled_loader = unlabeled_loaders[client_index]

    labeled_iterator = iter(labeled_loader)
    unlabeled_iterator = iter(unlabeled_loader)

    weight_u = unsupervised_weight(round_number)
    step_count = CLIENT_STEPS[client_index]

    supervised_total = 0.0
    unsupervised_total = 0.0
    proximal_total = 0.0

    for local_step in range(step_count):
        try:
            labeled_images, labeled_masks = next(labeled_iterator)
        except StopIteration:
            labeled_iterator = iter(labeled_loader)
            labeled_images, labeled_masks = next(labeled_iterator)


        try:
            unlabeled_images, _ = next(unlabeled_iterator)
        except StopIteration:
            unlabeled_iterator = iter(unlabeled_loader)
            unlabeled_images, _ = next(unlabeled_iterator)

        optimizer.zero_grad(set_to_none=True)

        labeled_count = labeled_images.shape[0]
        unlabeled_count = unlabeled_images.shape[0]

        step_supervised = 0.0
        step_unsupervised = 0.0


        for start in range(0, labeled_count, MICRO_BATCH_SIZE):
            end = min(start + MICRO_BATCH_SIZE, labeled_count)

            images = labeled_images[start:end].to(device)
            masks = labeled_masks[start:end].to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                logits = student(images)
                loss = F.binary_cross_entropy_with_logits(logits, masks)

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite supervised loss for client {client_index + 1}"
                )

            fraction = (end - start) / labeled_count
            scaler.scale(loss * fraction).backward()
            step_supervised += loss.item() * fraction

        if weight_u > 0:


            accepted_total = 0

            with torch.no_grad():
                for start in range(0, unlabeled_count, MICRO_BATCH_SIZE):
                    end = min(start + MICRO_BATCH_SIZE, unlabeled_count)
                    weak_images = unlabeled_images[start:end].to(device)

                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):
                        teacher_logits = teacher(weak_images)

                    probabilities = torch.sigmoid(teacher_logits.float())
                    confidence = torch.maximum(
                        probabilities,
                        1.0 - probabilities,
                    )
                    accepted_total += (
                        confidence >= CONFIDENCE_THRESHOLD
                    ).sum().item()

            for start in range(0, unlabeled_count, MICRO_BATCH_SIZE):
                end = min(start + MICRO_BATCH_SIZE, unlabeled_count)
                weak_images = unlabeled_images[start:end].to(device)

                with torch.no_grad():
                    with torch.autocast(
                        device_type="cuda",
                        dtype=torch.float16,
                    ):
                        teacher_logits = teacher(weak_images)

                    probabilities = torch.sigmoid(teacher_logits.float())
                    confidence = torch.maximum(
                        probabilities,
                        1.0 - probabilities,
                    )
                    accepted = confidence >= CONFIDENCE_THRESHOLD
                    pseudo_masks = (probabilities >= 0.5).float()

                strong_images = torch.stack([
                    strong_color_jitter(image)
                    for image in weak_images
                ]).clamp(0.0, 1.0)

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    student_logits = student(strong_images)
                    pixel_loss = F.binary_cross_entropy_with_logits(
                        student_logits.float(),
                        pseudo_masks,
                        reduction="none",
                    )

                    unsupervised_loss = (
                        pixel_loss * accepted.float()
                    ).sum() / max(accepted_total, 1)

                if not torch.isfinite(unsupervised_loss).item():
                    raise RuntimeError(
                        f"Non-finite SSL loss for client {client_index + 1}"
                    )

                scaler.scale(weight_u * unsupervised_loss).backward()
                step_unsupervised += unsupervised_loss.item()

        proximal_squared = torch.zeros((), device=device)

        for parameter, global_parameter in zip(
            student.parameters(), global_parameters
        ):
            proximal_squared = proximal_squared + (
                parameter - global_parameter
            ).square().sum()

        proximal_loss = 0.5 * MU * proximal_squared

        if not torch.isfinite(proximal_loss).item():
            raise RuntimeError(
                f"Non-finite proximal loss for client {client_index + 1}"
            )

        scaler.scale(proximal_loss).backward()
        scaler.unscale_(optimizer)

        for parameter in student.parameters():
            if parameter.grad is not None:
                if not torch.isfinite(parameter.grad).all().item():
                    raise RuntimeError(
                        f"Non-finite gradient for client {client_index + 1}"
                    )

        scaler.step(optimizer)
        scaler.update()

        global_step = (
            (round_number - 1) * sum(CLIENT_STEPS)
            + local_step + 1
        )
        ema_decay = min(
            MAX_EMA_DECAY,
            1.0 - 1.0 / (global_step + 1.0),
        )
        update_ema(teacher, student, ema_decay)

        supervised_total += step_supervised
        unsupervised_total += step_unsupervised
        proximal_total += proximal_loss.item()

    updated_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in student.state_dict().items()
    }

    metrics = {
        "supervised_loss": supervised_total / step_count,
        "unsupervised_loss": unsupervised_total / step_count,
        "proximal_loss": proximal_total / step_count,
        "steps": step_count,
    }

    del student, teacher, optimizer, global_parameters, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return updated_state, metrics






global_model = UNet().to(device)

if LAST_PATH.is_file():
    checkpoint = torch.load(
        LAST_PATH,
        map_location="cpu",
        weights_only=False,
    )

    assert checkpoint["config"] == run_config, (
        "Checkpoint settings differ from this run."
    )
    assert BEST_PATH.is_file(), "Best checkpoint is missing."

    global_model.load_state_dict(checkpoint["model_state"])
    completed_round = checkpoint["round"]
    best_iou = checkpoint["best_val_iou"]
    best_round = checkpoint["best_round"]
    history = list(checkpoint["history"])

    random.setstate(checkpoint["python_rng"])
    np.random.set_state(checkpoint["numpy_rng"])
    torch.set_rng_state(checkpoint["torch_rng"])
    torch.cuda.set_rng_state(checkpoint["cuda_rng"])

    for client_index in range(3):
        labeled_generators[client_index].set_state(
            checkpoint["labeled_generators"][client_index]
        )
        unlabeled_generators[client_index].set_state(
            checkpoint["unlabeled_generators"][client_index]
        )

    print(f"Resuming after round {completed_round}.")

else:
    assert not BEST_PATH.exists(), (
        "A best checkpoint exists without a last checkpoint."
    )

    set_seed(SEED)
    global_model = UNet().to(device)

    completed_round = 0
    best_iou = -1.0
    best_round = 0
    history = []

    print("Starting experiment B from round 1.")






for round_number in range(
    completed_round + 1,
    COMMUNICATION_ROUNDS + 1,
):
    round_start = time.time()

    global_state = {
        name: tensor.detach().cpu().clone()
        for name, tensor in global_model.state_dict().items()
    }

    client_states = []
    client_metrics = []

    print(
        f"\nRound {round_number}/{COMMUNICATION_ROUNDS} | "
        f"Unsupervised weight: {unsupervised_weight(round_number):.2f}",
        flush=True,
    )

    for client_index in range(3):
        client_start = time.time()

        state, metrics = train_client(
            global_state,
            client_index,
            round_number,
        )

        client_states.append(state)
        client_metrics.append(metrics)

        print(
            f"Client {client_index + 1} | "
            f"Supervised: {metrics['supervised_loss']:.4f} | "
            f"Unsupervised: {metrics['unsupervised_loss']:.4f} | "
            f"Proximal: {metrics['proximal_loss']:.4f} | "
            f"{time.time() - client_start:.1f}s",
            flush=True,
        )

    aggregated_state = aggregate_states_cpu(
        client_states,
        CLIENT_SIZES,
    )
    global_model.load_state_dict(aggregated_state)

    val_metrics = evaluate_unet(global_model, val_loader_small)

    improved = val_metrics["iou"] > best_iou

    if improved:
        best_iou = val_metrics["iou"]
        best_round = round_number

    row = {
        "round": round_number,
        "unsupervised_weight": unsupervised_weight(round_number),
        **{f"val_{key}": value for key, value in val_metrics.items()},
        **{
            f"client_{index + 1}_{key}": value
            for index, metrics in enumerate(client_metrics)
            for key, value in metrics.items()
        },
        "seconds": time.time() - round_start,
    }
    history.append(row)

    checkpoint = {
        "config": run_config,
        "round": round_number,
        "model_state": aggregated_state,
        "best_val_iou": best_iou,
        "best_round": best_round,
        "history": history.copy(),
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state(),
        "labeled_generators": {
            k: generator.get_state()
            for k, generator in labeled_generators.items()
        },
        "unlabeled_generators": {
            k: generator.get_state()
            for k, generator in unlabeled_generators.items()
        },
    }

    if improved:
        save_checkpoint_atomic(BEST_PATH, checkpoint)

    save_checkpoint_atomic(LAST_PATH, checkpoint)
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Val IoU: {val_metrics['iou']:.6f} | "
        f"Val F1: {val_metrics['f1']:.6f} | "
        f"Best: {best_iou:.6f} at round {best_round} | "
        f"{time.time() - round_start:.1f}s",
        flush=True,
    )
    print("Checkpoint saved.", flush=True)


print("\nTraining completed.")
print(f"Best validation IoU: {best_iou:.6f}")
print("Best round:", best_round)


best_checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

global_model.load_state_dict(best_checkpoint["model_state"])
test_metrics = evaluate_unet(global_model, test_loader_small)

print("\nHeld-out test results:")
for name, value in test_metrics.items():
    print(f"{name}: {value:.6f}")

result = {
    "experiment": run_config["experiment"],
    "selected_round": best_checkpoint["round"],
    "best_validation_iou": best_checkpoint["best_val_iou"],
    "test_metrics": test_metrics,
    "config": run_config,
}

import json

result_path = OUTPUT_DIR / "best_model_test_results.json"

with open(result_path, "w") as file:
    json.dump(result, file, indent=2)

print("\nSaved test results:", result_path)

GPU: Tesla T4
FedProx μ: 0.01
Physical batch size: 2
Effective batch size: 8
Results directory: /content/drive/MyDrive/DLP_Project/friend_unet_fedprox_student_teacher
Resuming after round 7.

Round 8/60 | Unsupervised weight: 0.30
Client 1 | Supervised: 0.3618 | Unsupervised: 0.1687 | Proximal: 0.0605 | 39.8s
Client 2 | Supervised: 0.3253 | Unsupervised: 0.1641 | Proximal: 0.0774 | 16.4s
Client 3 | Supervised: 0.3564 | Unsupervised: 0.1065 | Proximal: 0.0588 | 22.2s
Val IoU: 0.698816 | Val F1: 0.822710 | Best: 0.708665 at round 5 | 81.4s
Checkpoint saved.

Round 9/60 | Unsupervised weight: 0.40
Client 1 | Supervised: 0.4095 | Unsupervised: 0.1209 | Proximal: 0.0657 | 40.3s
Client 2 | Supervised: 0.3376 | Unsupervised: 0.1132 | Proximal: 0.0715 | 16.0s
Client 3 | Supervised: 0.3353 | Unsupervised: 0.1050 | Proximal: 0.0556 | 21.5s
Val IoU: 0.711112 | Val F1: 0.831170 | Best: 0.711112 at round 9 | 81.6s
Checkpoint saved.

Round 10/60 | Unsupervised weight: 0.50
Client 1 | Supervised: 0.3